Gate 1

# Project Setup & Modeling Contract

---

In [3]:
#Imports
from google.colab import drive
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from pathlib import Path

In [6]:
# ==============================================================================
# COLAB: Uncomment these lines to load from your Google Drive
# ==============================================================================

# drive.mount("/content/drive")
# df = pd.read_csv("/content/drive/MyDrive/allstate_claims_data.csv")

# ==============================================================================
# LOCAL/REPO: Keep the Colab lines above commented out and run this block
# ==============================================================================
# Load the dataset from this repo, on any computer.
data_path = None
for folder in [Path.cwd(), *Path.cwd().resolve().parents]:
    candidate = folder / "data" / "allstate_claims_data.csv"
    if candidate.is_file():
        data_path = candidate
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find data/allstate_claims_data.csv. "
        "Open this notebook from inside the cloned repository."
    )

df = pd.read_csv(data_path)

**Task & Baselines**
* **Prediction Task:** Estimate one claim's final paid amount (`loss`) from anonymous claim fields.
* **Metric:** Mean Absolute Error (MAE) evaluated in original `loss` units.
* **Baselines:** Training-mean constant baseline and a training-median constant comparator.

**Required Records & Approvals**
* **September Artifacts Link:** https://github.com/Break-Through-Tech/Allstate-1D-predicting-auto-claims-severity/tree/main/artifacts/september
* **Unresolved Anomalies:** None unresolved.
* **Recommendation Register:**
Adopted: Exploring an XGBoost model, random forest, use cross validation.
Defer: Remove highly correlated features

* **Prior Exposure:** Full-file EDA in September focused on exploring and plotting graphs of the dataset.

In [ ]:
#this freezes feature list and also exludes ID and loss
features = [col for col in df.columns if col not in ["loss", "id"]]
num_features = df[features].select_dtypes(include=np.number).columns.tolist()
cat_features = df[features].select_dtypes(exclude=np.number).columns.tolist()

# Evaluation and split design

---

In [ ]:
#test set is 15% of original data
train_temp, test_df = train_test_split(df, test_size=0.15, random_state=1)
#validatoin set is 15% of original data
train_df, val_df = train_test_split(train_temp, test_size=(0.15/0.85), random_state=1)

# Create machine readable partition manifest keyed by id
manifest = pd.concat([
    pd.DataFrame({"id": train_df["id"], 'partition': 'train'}),
    pd.DataFrame({"id": val_df["id"], 'partition': 'validation'}),
    pd.DataFrame({"id": test_df["id"], 'partition': 'test'})
])

# Export manifest
manifest.to_csv('gate1_partition_manifest.csv', index=False)
manifest

,id,partition
76216,238427,train
35306,110608,train
188234,587400,train
104048,325126,train
40719,127688,train
...,...,...
184254,574914,test
160613,500608,test
174650,545084,test
46865,147114,test


In [ ]:
#checks for overlapping ids
assert manifest["id"].nunique() == len(df)
#checks that the legnth of the manifest (rows) is equal to the original dataframe
assert len(manifest) == len(df)

In [ ]:
print(" Partition Row Counts")
print(manifest['partition'].value_counts())
print("\n")
print("Target (Loss) Summaries")
print("Train:\n", train_df["loss"].describe(percentiles=[.5, .9, .99]))
print("Validation:\n", val_df["loss"].describe(percentiles=[.5, .9, .99]))
print("\n")

#checks if there are unique values in validation data that are not in the training data
for cat in cat_features:
    train_levels = set(train_df[cat].unique())
    val_levels = set(val_df[cat].unique())
    unseen = val_levels - train_levels
    if unseen:
        print(f"Feature '{cat}' has {len(unseen)} unseen values in validation data.")

 Partition Row Counts
partition
train         131822
validation     28248
test           28248
Name: count, dtype: int64


Target (Loss) Summaries
Train:
 count    131822.000000
mean       3033.688044
std        2915.370042
min           0.670000
50%        2113.755000
90%        6382.559000
99%       14017.409800
max      121012.250000
Name: loss, dtype: float64
Validation:
 count    28248.000000
mean      3055.652346
std       2907.103183
min         54.340000
50%       2121.985000
90%       6467.678000
99%      14059.725200
max      51383.040000
Name: loss, dtype: float64


Feature 'cat92' has 1 unseen values in validation data.
Feature 'cat104' has 1 unseen values in validation data.
Feature 'cat105' has 2 unseen values in validation data.
Feature 'cat109' has 2 unseen values in validation data.
Feature 'cat110' has 1 unseen values in validation data.
Feature 'cat113' has 1 unseen values in validation data.
Feature 'cat114' has 1 unseen values in validation data.
Feature 'cat116' h

In [ ]:
# 1. Training-mean constant baseline
mean_pred = np.full(len(val_df), train_df["loss"].mean()) #calculates mean from training data
mean_mae = mean_absolute_error(val_df["loss"], mean_pred) #calculates MAE on validation data

# 2. Training-median constant comparator
median_pred = np.full(len(val_df), train_df["loss"].median()) #calculates median from training data
median_mae = mean_absolute_error(val_df["loss"], median_pred) #calculates MAE on validation data

print("Baseline Validation MAEs")

print(f"Mean Baseline MAE:   {mean_mae}")
print(f"Partition: Validation | Claim count: {len(val_df)} | Code: sklearn.metrics.mean_absolute_error\n")

print(f"Median Baseline MAE: {median_mae}")
print(f"Partition: Validation | Claim count: {len(val_df)} | Code: sklearn.metrics.mean_absolute_error")

Baseline Validation MAEs
Mean Baseline MAE:   1978.3970470120255
Partition: Validation | Claim count: 28248 | Code: sklearn.metrics.mean_absolute_error

Median Baseline MAE: 1827.3136338855847
Partition: Validation | Claim count: 28248 | Code: sklearn.metrics.mean_absolute_error


**Evaluation & Split Design**
* **Split Design:** 70% Train, 15% Validation, 15% Test. (Method: `train_test_split`, `random_state=1`).
* **Test Partition:** A 15% test partition is reserved strictly for November's final evaluation. Test targets will remain completely unseen during October fitting and selection.
* **Validation Usage & Stopping Rule:** Validation will be used to compare models, and we will not change seeds if validation MAE looks unfavorable. Iterations will stop when validation MAE fails to improve for 3 consecutive tuning runs.
* **Manifest Verification:** The partition manifest (`gate1_partition_manifest.csv`) has been frozen. Code checks confirmed partitions do not overlap or omit eligible claims, and target summaries/unseen categorical levels were logged.

# Preprocessing and experiment plan

---

**Preprocessing Plan**

* Imputers, scalers, and encoders will be fit on the training partition only to prevent data leakage.
* Numeric missing values will be filled in using the training median.
* Nominal categories will be One-Hot Encoded. To handle previously unseen or rare levels in the validation set, the encoder will be set to ignore them (handle_unknown='ignore').
* Target encoding is marked out of scope for the first comparison.

**Potential Model Approaches**

1. **Random Forest:**  A tree-based ensemble could capture non-linear relationships without requiring extensive scaling.
2. **XGBoost:**  Gradient boosting optimized directly for absolute error (`reg:absoluteerror`) can yield the lowest MAE.